# SEGUNDO PARCIAL

Enunciado

23. Diseña un agente para un juego de recolección de frutas con tiempo limitado.

Estrategia de Selección de acciones con intervalo de confianza (UCB)

1. Entorno: (tablero-cuadricula) con límite de tiempo y frutas.

Estado (s): Sera la posición actual del agente (X, Y).

Acciones (A): Arriba, Abajo, Izquierda, Derecha.

Recompensas (R): +10 por comer fruta -0.1 por cada paso (para que pueda ser rápido).

Política: Las reglas para moverse (basado en la estrategia UCB). Función Q unna tabla que guarda el puntaje estimado de que tan buena es una acción en una posición específica

2. Estrategia UCB (Exploración Inteligente)En lugar de probar movimientos al azar a ciegas, UCB calcula un (bono de curiosidad) para las acciones que el agente menos conoce, se elige la acción que maximiza esta fórmula:

$$
    A_t = \underset{a}{\arg\max} \, \left[ Q_t(a) + c \sqrt{\frac{\text{ln} \, t}{N_t(a)}} \right]
$$


## Entorno

In [7]:
import numpy as np

class EntornoFrutas():
    def __init__(self, tamanio=4, limite_tiempo=12):
        self.tamano = tamanio
        self.limite_tiempo = limite_tiempo
        # fijams las frutas iniciales
        self.frutas_iniciales = [(1, 1), (2, 3), (3, 0)]
        self.reset()

    def valid_moves(self):
        # acciones (0: arriba, 1; abaj, 2: izquierda, 3: derecha)
        moves = []
        r, c = self.posicion
        if r > 0: moves.append(0)
        if r < self.tamano - 1: moves.append(1)
        if c > 0: moves.append(2)
        if c < self.tamano - 1: moves.append(3)
        return moves

    def update(self, action):
        
        self.tiempo += 1
        r, c = self.posicion
        
        if action == 0: r -= 1
        elif action == 1: r += 1
        elif action == 2: c -= 1
        elif action == 3: c += 1
            
        self.posicion = (r, c)
        
        # recolectar fruta
        if self.posicion in self.frutas:
            self.frutas.remove(self.posicion)

    def is_game_over(self):
        # comprobar 3 en raya
        if len(self.frutas) == 0:
            return 1 # victoria (1)
        if self.tiempo >= self.limite_tiempo:
            return 0 # derrota (0)
        return None 

    def estado_actual(self):
        return str((self.posicion, tuple(self.frutas)))

    def reset(self):
        self.posicion = (0, 0)
        self.tiempo = 0
        self.frutas = self.frutas_iniciales.copy()

## Juego

In [9]:
from tqdm import tqdm

class JuegoFrutas():
    def __init__(self, agente):
        self.agente = agente
        self.entorno = EntornoFrutas()

    def play(self, rounds=1000):
        wins = 0
        for i in tqdm(range(1, rounds + 1)):
            self.entorno.reset()
            self.agente.reset()
            game_over = False
            
            while not game_over:
                # agente debe actualizar entorno?
                action = self.agente.move(self.entorno)
                self.entorno.update(action)
                self.agente.update(self.entorno)
                
                if self.entorno.is_game_over() is not None:
                    game_over = True
                    break
                    
            self.reward()
            if self.entorno.is_game_over() == 1:
                wins += 1
        return wins

    def reward(self):
        #recompensa al final de la partida
        ganador = self.entorno.is_game_over()
        if ganador == 1: 
            self.agente.reward(1.0) # premi por limpiar el tablero
        else: 
            self.agente.reward(0.0) # perdip por tiempo

## UCB

In [10]:
import math

class AgenteUCB():
    def __init__(self, alpha=0.5, c=1.5):
        self.value_function = {} 
        self.n_function = {}     
        self.alpha = alpha         
        self.c = c                 
        self.positions = []       
        self.t = 0               

    def reset(self):
        self.positions = []

    def move(self, board):
        valid_moves = board.valid_moves()
        self.t += 1
        
        max_ucb = -10000
        best_action = valid_moves[0]
        
        for action in valid_moves:
            
            next_board = copy.deepcopy(board)
            next_board.update(action)
            next_state = next_board.estado_actual()
            
            
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)
            visitas = 0 if self.n_function.get(next_state) is None else self.n_function.get(next_state)
            
            
            if visitas == 0:
                
                return action
                
            
            ucb_score = value + self.c * math.sqrt(math.log(self.t) / visitas)
            
            if ucb_score >= max_ucb:
                max_ucb = ucb_score
                best_action = action
                
        return best_action

    def update(self, board):
        
        estado = board.estado_actual()
        self.positions.append(estado)
        self.n_function[estado] = self.n_function.get(estado, 0) + 1

    def reward(self, reward):
        
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

## Entrenamiento

In [12]:

agente = AgenteUCB(alpha=0.5, c=1.5)

juego = JuegoFrutas(agente)
victorias = juego.play(10000)
print(f"\\nPartidas ganadas: {victorias} de 10000")

100%|██████████| 10000/10000 [00:06<00:00, 1435.67it/s]

\nPartidas ganadas: 8691 de 10000
